In [2]:
import socket

In [3]:
import threading

In [ ]:
import socket
import threading
import time

HOST = "127.0.0.1"
PORT = 65002
MAX_WORKERS = 2

worker_slots = threading.Semaphore(MAX_WORKERS)


def handle_client(conn, addr):

    with conn:

        request = conn.recv(1024).decode()

        print(
            f"[WORKER-{threading.get_ident()}] "
            f"Waiting for a free slot ({addr})"
        )

        # Worker waits here if both slots are busy
        with worker_slots:

            print(
                f"[WORKER-{threading.get_ident()}] "
                f"Got a slot, working on {addr}"
            )

            # Simulate 2 seconds of work
            time.sleep(2)

            reply = (
                f"Processed '{request}' "
                f"by worker thread {threading.get_ident()}"
            )

            conn.sendall(reply.encode())

            print(
                f"[WORKER-{threading.get_ident()}] "
                f"Done with {addr}, releasing slot"
            )


def main():

    with socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    ) as server_socket:

        server_socket.setsockopt(
            socket.SOL_SOCKET,
            socket.SO_REUSEADDR,
            1
        )

        server_socket.bind((HOST, PORT))

        server_socket.listen(5)

        print(
            f"[DISPATCHER] Listening on {HOST}:{PORT} "
            f"(max {MAX_WORKERS} concurrent workers)"
        )

        while True:

            # Dispatcher accepts connection immediately
            conn, addr = server_socket.accept()

            print(
                f"[DISPATCHER] Accepted connection from {addr}, "
                f"spawning worker thread"
            )

            # Create worker immediately
            worker = threading.Thread(
                target=handle_client,
                args=(conn, addr)
            )

            worker.start()


if __name__ == "__main__":
    main()

[DISPATCHER] Listening on 127.0.0.1:65002 (max 2 concurrent workers)
